# Test synchronisation Docs → Grist
Notebook de test pour valider chaque étape avant la synchro complète.

In [1]:
import sys
sys.path.insert(0, '../src')

from dotenv import load_dotenv
load_dotenv('../.env')

True

## 1. Connexion au client Docs
Renseigne `DOCS_SESSION_ID` et `DOCS_CSRF_TOKEN` dans le fichier `.env` avant de lancer cette cellule.

In [2]:
import os
from docs_client import DocsClient

docs = DocsClient(
    base_url='https://docs.numerique.gouv.fr',
    session_id=os.environ.get('DOCS_SESSION_ID'),
    csrf_token=os.environ.get('DOCS_CSRF_TOKEN'),
)
print('DocsClient initialisé.')

DocsClient initialisé.


## 2. Récupération du tree
Remplace l'URL ci-dessous par celle de ton document racine.

In [3]:
ROOT_URL = 'https://docs.numerique.gouv.fr/docs/21bb0cdd-638f-4a50-8f3a-03616716771e/'

doc_id = DocsClient.extract_doc_id(ROOT_URL)
tree = docs.get_tree(doc_id)

print(f"Racine : {tree['title']}")
print(f"Enfants directs : {len(tree.get('children', []))}")

Racine : Cartographie Stratégique du Bruit - Guide Méthodologique
Enfants directs : 13


## 3. Aplatissement du tree en records Grist

In [4]:

# ── Choix du mode de récupération du contenu ─────────────────────────────────
# 'json'     → JSON BlockNote → markdown (défaut, préserve callouts et blocs custom)
# 'markdown' → markdown brut (peut échouer sur certains blocs BlockNote custom)
# 'html'     → HTML converti en texte → plus fiable pour les émojis du corps
# 'auto'     → essaie markdown, bascule HTML si le Y-Provider plante
CONTENT_FORMAT = 'json'

records = docs.flatten_tree(
    tree,
    'https://docs.numerique.gouv.fr',
    is_root=True,
    content_format=CONTENT_FORMAT,
)

print(f'\n── {len(records)} records au total ──')


[racine] Cartographie Stratégique du Bruit - Guide Méthodologique
  [1] 🌟 Guide
    [1.1] 📜 ​Rapportage
    ✓ 1 émoji(s) dans le contenu [json→md]
    [1.2] 🛣️ GITT
    ✓ 1 émoji(s) dans le contenu [json→md]
    [1.3] 🗺️ CBS
    [1.4] 🔠 Glossaire
    [1.5] 📚 Bibliographie associée
    [1.6] ℹ️ À propos du guide
  [2] 🛠️ Outils
    [2.1] 🔊 NoiseModelling
    [2.2] 💽 Bases de données
    [2.3] 🖥️ Serveur de calcul
  [3] 🏘️ ​🔴 Bâtiments
    ✓ 2 émoji(s) dans le contenu [json→md]
    [3.1] 🙎‍♂️ ​Affectation des populations
    ✓ 1 émoji(s) dans le contenu [json→md]
    [3.2] 🧑‍🔧 Implémentation - Bâti
  [4] 📍 Récepteurs
    [4.1] 👩‍👦 ​🔵 Décompte de population
    ✓ 1 émoji(s) dans le contenu [json→md]
    [4.2] 🎨 Visuel - isosurfaces
    ✓ 1 émoji(s) dans le contenu [json→md]
    [4.3] 🧑‍🔧 ​🔵 Implémentation - Récepteurs
  [5] 🚙 Routier
    [5.1] 🌐 ​🔴 Géométrie
    ✓ 1 émoji(s) dans le contenu [json→md]
    [5.2] 📊 ​🔴 Débits
    [5.3] ⏩ ​Vitesses
    ✓ 1 émoji(s) dans le contenu [json→md]
  

## 4. Envoi vers Grist
⚠️ Lance cette cellule uniquement quand les records semblent corrects.

In [5]:

from grist_client import GristClient
from docs_client import DocsClient

grist = GristClient()

# ── Colonnes existantes dans la table Grist ───────────────────────────────────
GRIST_COLUMNS = {'titre', 'niveau', 'ordre', 'numero', 'url', 'contenu'}
# GRIST_COLUMNS = None  # ← décommente pour envoyer tous les champs sans filtrage

records_clean = [
    {**r, "fields": {
        **r["fields"],
        "contenu": DocsClient.sanitize_for_waf(r["fields"].get("contenu", ""))
        if isinstance(r["fields"].get("contenu"), str) else r["fields"].get("contenu", "")
    }}
    for r in records
]

ok, ko = grist.send_records('Chapitres', records_clean, columns=GRIST_COLUMNS)


  [  1/50] ✓  🌟 Guide  (605 car.)
  [  2/50] ✓  📜 ​Rapportage  (3575 car.)
  [  3/50] ✓  🛣️ GITT  (2335 car.)
  [  4/50] ✓  🗺️ CBS  (811 car.)
  [  5/50] ✓  🔠 Glossaire  (1179 car.)
  [  6/50] ✓  📚 Bibliographie associée  (507 car.)
  [  7/50] ✓  ℹ️ À propos du guide  (362 car.)
  [  8/50] ✓  🛠️ Outils  (245 car.)
  [  9/50] ✓  🔊 NoiseModelling  (771 car.)
  [ 10/50] ✓  💽 Bases de données  (936 car.)
  [ 11/50] ✓  🖥️ Serveur de calcul  (953 car.)
  [ 12/50] ✓  🏘️ ​🔴 Bâtiments  (6717 car.)
  [ 13/50] ✓  🙎‍♂️ ​Affectation des populations  (3063 car.)
  [ 14/50] ✓  🧑‍🔧 Implémentation - Bâti  (1142 car.)
  [ 15/50] ✓  📍 Récepteurs  (1237 car.)
  [ 16/50] ✓  👩‍👦 ​🔵 Décompte de population  (6495 car.)
  [ 17/50] ✓  🎨 Visuel - isosurfaces  (2918 car.)
  [ 18/50] ✓  🧑‍🔧 ​🔵 Implémentation - Récepteurs  (5969 car.)
  [ 19/50] ✓  🚙 Routier  (462 car.)
  [ 20/50] ✓  🌐 ​🔴 Géométrie  (1543 car.)
  [ 21/50] ✓  📊 ​🔴 Débits  (4506 car.)
  [ 22/50] ✓  ⏩ ​Vitesses  (1891 car.)
  [ 23/50] ✓  ✖️ Intersecti

### 4.1. Annexe : test d'un record
🧪 Cellule pour tester un record précis

In [10]:

# ── Envoi individuel d'un record (debug — cellule autonome) ──────────────────
from grist_client import GristClient
from docs_client import DocsClient

INDEX = 21  # numéro du record à tester (base 1)

_GRIST_COLUMNS = {'titre', 'niveau', 'ordre', 'numero', 'url', 'contenu'}

_grist = GristClient()
_record = records[INDEX - 1]

_contenu = _record["fields"].get("contenu", "")
if isinstance(_contenu, str):
    _contenu = DocsClient.sanitize_for_waf(_contenu)
_record_clean = {**_record, "fields": {**_record["fields"], "contenu": _contenu}}

print(f"Titre          : {_record_clean['fields'].get('titre')!r}")
print(f"Taille contenu : {len(_contenu)} car.")
print()

ok, ko = _grist.send_records('Chapitres', [_record_clean], columns=_GRIST_COLUMNS)


Champs envoyés : ['titre', 'niveau', 'ordre', 'numero', 'url', 'contenu']
Taille contenu : 4411 car.
Titre          : '📊 \u200b🔴 Débits'

✓ Record 21 ajouté. Réponse : {'records': [{'id': 50}]}
